In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np, pandas as pd, GPy, warnings

warnings.filterwarnings("ignore")

GPy.util.linalg.jitchol_MAXTRIES = 20

ROOT_FOLDER = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/FPGA DATA/Raw-data_exahaustive"
OUT_FOLDER = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/svts"
FPGA_START, FPGA_END = 1, 50
NUM_PATHS = 32
TRAIN_RATIO = 0.10
SEED = 42
MODE = "all"  # "single" or "all"
ALPHA, BETA = 0.7, 0.3  # paper's best config
CALIB_FPGA = 1


def load_dat(path):
    grid = np.loadtxt(path, delimiter=",")
    rows, cols = grid.shape
    rec = [
        (x, y, grid[y, x])
        for y in range(rows)
        for x in range(cols)
        if grid[y, x] != 0.0
    ]
    df = pd.DataFrame(rec, columns=["X", "Y", "Value"])
    median, std = df.Value.median(), df.Value.std()
    return df[np.abs(df.Value - median) <= 3 * std].reset_index(drop=True)


def select_train_coords(df):
    X = df[["X", "Y"]].values.astype(float)
    Y = df["Value"].values.astype(float)
    n = len(Y)
    n_train = int(n * TRAIN_RATIO)
    rng = np.random.default_rng(SEED)
    idx_all = np.arange(n)
    train_idx = [rng.choice(idx_all)]
    while len(train_idx) < n_train:
        remaining = np.setdiff1d(idx_all, train_idx)
        m = GPy.models.GPRegression(
            X[train_idx], Y[train_idx].reshape(-1, 1), GPy.kern.RBF(2), normalizer=True
        )
        m.optimize(messages=False)
        mean_r, var_r = m.predict(X[remaining])
        mean_s, var_s = m.predict(X[train_idx])

        all_mean = np.concatenate([mean_r.ravel(), mean_s.ravel()])
        all_var = np.concatenate([var_r.ravel(), var_s.ravel()])
        mean_z = (all_mean - all_mean.mean()) / (all_mean.std() + 1e-9)
        var_z = (all_var - all_var.mean()) / (all_var.std() + 1e-9)
        n_r = len(mean_r)
        feat_r = np.column_stack([mean_z[:n_r], var_z[:n_r]])
        feat_s = np.column_stack([mean_z[n_r:], var_z[n_r:]])

        centroid = feat_s.mean(axis=0)
        centroid_c = centroid - centroid.mean()
        feat_r_c = feat_r - feat_r.mean(axis=1, keepdims=True)
        sim = (feat_r_c @ centroid_c) / (
            np.linalg.norm(feat_r_c, axis=1) * np.linalg.norm(centroid_c) + 1e-9
        )
        sim_n = (sim - sim.min()) / (sim.max() - sim.min() + 1e-9)
        var_n = (var_r.ravel() - var_r.min()) / (var_r.max() - var_r.min() + 1e-9)

        score = ALPHA * sim_n + BETA * var_n
        train_idx.append(remaining[np.argmax(score)])
    return X[train_idx]


def svts(df, train_coords):
    X = df[["X", "Y"]].values.astype(float)
    Y = df["Value"].values.astype(float)
    is_train = (X[:, None, :] == train_coords[None, :, :]).all(axis=2).any(axis=1)
    Xtr, Ytr = X[is_train], Y[is_train]
    Xte, Yte = X[~is_train], Y[~is_train]
    m = GPy.models.GPRegression(
        Xtr, Ytr.reshape(-1, 1), GPy.kern.RBF(2), normalizer=True
    )
    m.optimize(messages=False)
    pred, var = m.predict(Xte)
    return pd.DataFrame(
        {
            "X": Xte[:, 0],
            "Y": Xte[:, 1],
            "VALUE": Yte,
            "PREDICTED VALUE": pred.ravel(),
            "VARIANCE": var.ravel(),
        }
    )


calib_positions = {}


def get_calib_coords(path):
    if path not in calib_positions:
        df = load_dat(f"{ROOT_FOLDER}/FPGA-{CALIB_FPGA}/data{path}.dat")
        calib_positions[path] = select_train_coords(df)
    return calib_positions[path]


fpga_range = range(FPGA_START, FPGA_END + 1) if MODE == "all" else [FPGA_START]
path_range = range(1, NUM_PATHS + 1) if MODE == "all" else [1]

for fpga in fpga_range:
    os.makedirs(f"{OUT_FOLDER}/FPGA{fpga}", exist_ok=True)
    for path in path_range:
        train_coords = get_calib_coords(path)
        df = load_dat(f"{ROOT_FOLDER}/FPGA-{fpga}/data{path}.dat")
        result = svts(df, train_coords)
        result.to_csv(f"{OUT_FOLDER}/FPGA{fpga}/path{path}_svts.csv", index=False)
        print(f"FPGA{fpga} path{path}_svts.csv saved")


FPGA1 path1_svts.csv saved
FPGA1 path2_svts.csv saved
FPGA1 path3_svts.csv saved
FPGA1 path4_svts.csv saved
FPGA1 path5_svts.csv saved
FPGA1 path6_svts.csv saved
FPGA1 path7_svts.csv saved
FPGA1 path8_svts.csv saved
FPGA1 path9_svts.csv saved
FPGA1 path10_svts.csv saved
FPGA1 path11_svts.csv saved
FPGA1 path12_svts.csv saved
FPGA1 path13_svts.csv saved
FPGA1 path14_svts.csv saved
FPGA1 path15_svts.csv saved
FPGA1 path16_svts.csv saved
FPGA1 path17_svts.csv saved
FPGA1 path18_svts.csv saved
FPGA1 path19_svts.csv saved
FPGA1 path20_svts.csv saved
FPGA1 path21_svts.csv saved
FPGA1 path22_svts.csv saved
FPGA1 path23_svts.csv saved
FPGA1 path24_svts.csv saved
FPGA1 path25_svts.csv saved
FPGA1 path26_svts.csv saved
FPGA1 path27_svts.csv saved
FPGA1 path28_svts.csv saved
FPGA1 path29_svts.csv saved
FPGA1 path30_svts.csv saved
FPGA1 path31_svts.csv saved
FPGA1 path32_svts.csv saved
FPGA2 path1_svts.csv saved
FPGA2 path2_svts.csv saved
FPGA2 path3_svts.csv saved
FPGA2 path4_svts.csv saved
FPGA2

In [2]:
import os

OUT_FOLDER = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/svts"


for fpga_dir in os.listdir(OUT_FOLDER):
    fpga_path = os.path.join(OUT_FOLDER, fpga_dir)
    n = fpga_dir.replace("FPGA", "")
    for f in os.listdir(fpga_path):
        if f.startswith("path") and f.endswith("_svts.csv"):
            new_name = f"fpga{n}_{f}"
            os.rename(os.path.join(fpga_path, f), os.path.join(fpga_path, new_name))


In [3]:
import os, re, pandas as pd

OUT_FOLDER = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/svts"

dfs = []
for fpga_dir in os.listdir(OUT_FOLDER):
    fpga_path = os.path.join(OUT_FOLDER, fpga_dir)
    for f in os.listdir(fpga_path):
        if f.endswith("_svts.csv"):
            path = int(re.findall(r"path(\d+)", f)[0])
            fpga = int(re.findall(r"\d+", fpga_dir)[0])
            df = pd.read_csv(os.path.join(fpga_path, f))[
                ["X", "Y", "VALUE", "PREDICTED VALUE", "VARIANCE"]
            ]
            df["FPGA"], df["PATH"] = int(fpga), int(path)
            dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
combined = combined.sort_values(["FPGA", "PATH"]).reset_index(drop=True)
combined.to_csv(f"{OUT_FOLDER}/combined_svts.csv", index=False)


In [4]:
import pandas as pd

OUT_FOLDER = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/svts"

df = pd.read_csv(f"{OUT_FOLDER}/combined_svts.csv")
df = df.sort_values(["FPGA", "PATH"]).reset_index(drop=True)
df.to_csv(f"{OUT_FOLDER}/combined_svts.csv", index=False)


In [5]:
import numpy as np, pandas as pd

ROOT_FOLDER = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/FPGA DATA/Raw-data_exahaustive"
OUT_FOLDER = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/svts"
FPGA_START, FPGA_END = 1, 50
NUM_PATHS = 32


def load_dat(fp):
    grid = np.loadtxt(fp, delimiter=",")
    rows, cols = grid.shape
    vals = [grid[y, x] for y in range(rows) for x in range(cols) if grid[y, x] != 0.0]
    vals = pd.Series(vals)
    vals = vals[np.abs(vals - vals.median()) <= 3 * vals.std()]
    return vals


rows = []
for fpga in range(FPGA_START, FPGA_END + 1):
    for path in range(1, NUM_PATHS + 1):
        vals = load_dat(f"{ROOT_FOLDER}/FPGA-{fpga}/data{path}.dat")
        rows.append(
            {
                "FPGA": fpga,
                "PATH": path,
                "Max Value": vals.max(),
                "Min Value": vals.min(),
                "dspec": vals.max() - vals.min(),
            }
        )

dspec_df = pd.DataFrame(rows)

df = pd.read_csv(f"{OUT_FOLDER}/combined_svts.csv")
df = df.merge(dspec_df, on=["FPGA", "PATH"], how="left")
df.to_csv(f"{OUT_FOLDER}/combined_svts.csv", index=False)


In [6]:
import pandas as pd

OUT_FOLDER = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/svts"

df = pd.read_csv(f"{OUT_FOLDER}/combined_svts.csv")
df["delta"] = (df["PREDICTED VALUE"] - df["VALUE"]) / df["dspec"]

per_path = (
    df.groupby(["FPGA", "PATH"])["delta"].apply(lambda x: x.abs().mean()).reset_index()
)
per_path.columns = ["FPGA", "PATH", "Delta"]
per_path.to_csv(f"{OUT_FOLDER}/svts_delta_per_path.csv", index=False)

per_fpga = per_path.groupby("FPGA")["Delta"].mean().reset_index()
per_fpga.to_csv(f"{OUT_FOLDER}/svts_delta_per_fpga.csv", index=False)

overall = per_fpga["Delta"].mean()
pd.DataFrame({"Overall Delta": [overall]}).to_csv(
    f"{OUT_FOLDER}/svts_delta_overall.csv", index=False
)
print(f"Overall Delta: {overall:.6f}")


Overall Delta: 0.095373


In [8]:
import os, numpy as np, pandas as pd

DATA_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/FPGA DATA/Raw-data_exahaustive"
TEST_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/svts/FPGA1"
OUT_ROOT = r"C:/Users/sysdp/OneDrive - 島根大学/Desktop/Research/MY PAPERS/Experiment 2/svts/FPGA1T"

FPGA = 1
NUM_PATHS = 32

os.makedirs(OUT_ROOT, exist_ok=True)

for path in range(1, NUM_PATHS + 1):
    # Load full raw data
    grid = np.loadtxt(
        os.path.join(DATA_ROOT, f"FPGA-{FPGA}", f"data{path}.dat"), delimiter=","
    )
    rows, cols = grid.shape
    rec = [
        (x, y, grid[y, x])
        for y in range(rows)
        for x in range(cols)
        if grid[y, x] != 0.0
    ]
    full = pd.DataFrame(rec, columns=["X", "Y", "Value"])
    median, std = full.Value.median(), full.Value.std()
    full = full[np.abs(full.Value - median) <= 3 * std].reset_index(drop=True)

    # Load test points CSV
    test_file = os.path.join(TEST_ROOT, f"fpga{FPGA}_path{path}_svts.csv")
    if not os.path.exists(test_file):
        print(f"Path-{path}: test file not found, skipping.")
        continue
    test = pd.read_csv(test_file)[["X", "Y"]]

    # Training points = full data minus test points
    test_set = set(zip(test.X, test.Y))
    train = full[~full.apply(lambda r: (r.X, r.Y) in test_set, axis=1)]

    train.to_csv(
        os.path.join(OUT_ROOT, f"fpga{FPGA}_path{path}_train_svts.csv"), index=False
    )
    print(f"Path-{path}: {len(train)} training points saved.")


Path-1: 315 training points saved.
Path-2: 317 training points saved.
Path-3: 317 training points saved.
Path-4: 317 training points saved.
Path-5: 317 training points saved.
Path-6: 317 training points saved.
Path-7: 317 training points saved.
Path-8: 317 training points saved.
Path-9: 317 training points saved.
Path-10: 317 training points saved.
Path-11: 317 training points saved.
Path-12: 317 training points saved.
Path-13: 317 training points saved.
Path-14: 317 training points saved.
Path-15: 317 training points saved.
Path-16: 317 training points saved.
Path-17: 317 training points saved.
Path-18: 317 training points saved.
Path-19: 317 training points saved.
Path-20: 317 training points saved.
Path-21: 317 training points saved.
Path-22: 317 training points saved.
Path-23: 317 training points saved.
Path-24: 317 training points saved.
Path-25: 317 training points saved.
Path-26: 317 training points saved.
Path-27: 317 training points saved.
Path-28: 317 training points saved.
P